# FHIR Search Validator Demo

This notebook demonstrates the extracted `fhir_validator_agent` utility validating FHIR queries against public FHIR servers.

We test both positive (valid) and negative (invalid) scenarios across HAPI FHIR R4 and Firely Public Server.

In [4]:
from fhir_validator_agent import FhirValidatorService

# Backward-compatible alias
FhirValidatorAgent = FhirValidatorService

## Test Configuration

Define test queries and public servers to validate against.

In [5]:
from fhir_validator_agent.config.public_servers import get_public_test_servers_without_auth

public_servers = get_public_test_servers_without_auth()

test_queries = [
    {"query": "Patient?gender=male", "expected": "positive"},  # Valid gender
    {"query": "Patient?gender=fe", "expected": "negative"},  # Invalid gender value
]

print(f"Testing {len(test_queries)} queries across {len(public_servers)} public servers (no auth)...")
for server in public_servers:
    print(f"  - {server['key']}: {server['base_url']}")

Testing 2 queries across 2 servers...


## Run Validation Tests

For each server, load the CapabilityStatement and validate both positive and negative test cases.

In [6]:
for server in public_servers:
    try:
        print(f"\n--- Testing with {server['name']} ---")
        
        # Create agent for this server
        agent = FhirValidatorAgent(
            metadata_url=server['metadata_url'],
            server_base=server['base_url']
        )
        
        # Run each test query
        for test in test_queries:
            print(f"\nTesting query: {test['query']} (expected: {test['expected']})")
            
            full_query_url = f"{server['base_url']}/{test['query']}"
            result = agent.validate_query(full_query_url)
            
            if result['valid']:
                print("Query is valid.")
            else:
                print("Validation errors:")
                for err in result['errors']:
                    print(f" - {err}")
                print("Query is invalid.")
    
    except Exception as e:
        print(f"Error with {server['name']}: {e}")


--- Testing with HAPI FHIR R4 ---

Testing query: Patient?gender=male (expected: positive)
Query is valid.

Testing query: Patient?gender=fe (expected: negative)
Validation errors:
 - Value 'fe' for 'Patient.gender' is not allowed. Allowed values: {'other', 'male', 'unknown', 'female'}
Query is invalid.

--- Testing with Firely Public FHIR Server ---

Testing query: Patient?gender=male (expected: positive)
Query is valid.

Testing query: Patient?gender=fe (expected: negative)
Validation errors:
 - Value 'fe' for 'Patient.gender' is not allowed. Allowed values: {'other', 'male', 'unknown', 'female'}
Query is invalid.


## Expected Results

### Positive Test Case: `Patient?gender=male`
- **Server**: HAPI FHIR & Firely
- **Query**: Patient resource search with gender parameter
- **Expected**: ✓ Valid - Both servers support the `gender` search parameter for Patient resources
- **Validation**: Passes all checks (valid resource type, valid parameter, standard FHIR parameter)

### Negative Test Case: `Patient?gender=fe`
- **Server**: HAPI FHIR & Firely  
- **Query**: Patient resource search with invalid gender value
- **Expected**: ✗ Invalid - The value `fe` is not a valid FHIR gender code
- **Validation**: Fails because `fe` is not in the standard FHIR value set `{male, female, other, unknown}`
- **Error Message**: "Invalid value 'fe' for parameter 'gender'. Allowed values: male, female, other, unknown"

### How to Read the Output
Each test displays:
1. **Server Name** - Which FHIR server was tested
2. **Query** - The FHIR search query being validated
3. **Expected Result** - What we expect (valid or invalid)
4. **Validation Status** - Whether the query passed or failed
5. **Error Details** - Specific reasons why invalid queries failed